# $S^2\times S^3$ 上のアインシュタイン計量

「$S^2\times S^3$ のリーマン曲率テンソル」という名前で公開されてきた
Egisonのデモは、2つの丸い計量の単純なブロック直積ではなく、非対角な
5次元計量を扱います。その目標は、次のアインシュタイン条件を確かめる
ことです。

$$
\operatorname{Ric}_{ij}=4g_{ij},\qquad \mathcal R=20.
$$

このNotebookでは、その計量を引き継ぎつつ、記号計算に必要な計算量を
明示します。計量や接続の小さな成分は対話的な出力に適していますが、
アインシュタイン条件の残差全体は、必要に応じて明示的に評価できるよう
定義だけを用意します。


## 座標と略記

座標の順序を $x=(\phi,\theta,\psi,y,\alpha)$ とし、次の略記を導入します。

$$
p=1-y,\quad u=a-y^2,\quad
v=a-3y^2+2y^3,\quad q=a-2y+y^2.
$$

これらの因子によって、元の計量に繰り返し現れる構造が見やすくなり、
座標領域の制約（$p u v\ne0$）も明確になります。


In [1]:
declare symbol φ, θ, ψ, y, α, a: MathValue

def x : Vector MathValue := [| φ, θ, ψ, y, α |]

def p : MathValue := `(1 - y)
def u : MathValue := `(a - y^2)
def v : MathValue := `(a - 3 * y^2 + 2 * y^3)
def q : MathValue := `(a - 2 * y + y^2)


In [2]:
(p, u, v, q)


$((-y + 1), (-y^{2} + a), (2 y^{3} - 3 y^{2} + a), (y^{2} - 2 y + a))$

## 計量

ゼロでない成分は、それぞれの対称成分と合わせて次の通りです。

$$
\begin{aligned}
g_{11}&=\frac{3p^2u\sin^2\theta+2vp\cos^2\theta+q^2\cos^2\theta}{18up},
&g_{13}&=\frac{-2vp\cos\theta-q^2\cos\theta}{18up},\\
g_{15}&=-\frac{q\cos\theta}{3p},
&g_{22}&=\frac p6,\\
g_{33}&=\frac{2vp+q^2}{18up},
&g_{35}&=\frac q{3p},\\
g_{44}&=\frac p{2v},
&g_{55}&=\frac{2u}{p}.
\end{aligned}
$$

$(1,3,5)$ ブロックは互いに結合しています。このため、逆計量全体の計算や
曲率の展開は、丸い球面・超球面を扱うNotebookよりも大幅に重くなります。


In [3]:
def g_i_j : Matrix MathValue :=
  [| [| (3 * p^2 * (sin θ)^2 * u + 2 * v * p * (cos θ)^2 + q^2 * (cos θ)^2) / (18 * u * p)
       , 0
       , (-2 * v * p * cos θ - q^2 * cos θ) / (18 * u * p)
       , 0
       , (-q * cos θ) / (3 * p) |]
   , [| 0, p / 6, 0, 0, 0 |]
   , [| (-2 * v * p * cos θ - q^2 * cos θ) / (18 * u * p)
       , 0
       , (2 * v * p + q^2) / (18 * u * p)
       , 0
       , q / (3 * p) |]
   , [| 0, 0, 0, p / (2 * v), 0 |]
   , [| (-q * cos θ) / (3 * p), 0, q / (3 * p), 0, 2 * u / p |]
   |]_i_j


In [4]:
g_2_2


$\frac{1}{6} (-y + 1)$

In [5]:
g_4_4


$\frac{1}{2} (-y + 1) (2 y^{3} - 3 y^{2} + a)^{-1}$

In [6]:
(g_1_3, g_3_1, g_1_5, g_5_1)


$(\frac{-1}{9} \cos(θ) (-y^{2} + a)^{-1} (2 y^{3} - 3 y^{2} + a) + \frac{-1}{18} \cos(θ) (-y + 1)^{-1} (-y^{2} + a)^{-1} (y^{2} - 2 y + a)^{2}, \frac{-1}{9} \cos(θ) (-y^{2} + a)^{-1} (2 y^{3} - 3 y^{2} + a) + \frac{-1}{18} \cos(θ) (-y + 1)^{-1} (-y^{2} + a)^{-1} (y^{2} - 2 y + a)^{2}, \frac{-1}{3} \cos(θ) (-y + 1)^{-1} (y^{2} - 2 y + a), \frac{-1}{3} \cos(θ) (-y + 1)^{-1} (y^{2} - 2 y + a))$

## 接続と曲率の計算手順

第一種クリストッフェル記号には、上で示した計量の微分だけが必要です。
添字を上げる段階で、結合したブロックの逆行列が現れます。それでも、
リーマンテンソルとリッチテンソルの定義は低次元の例と同じです。

$$
R^i{}_{jkl}=\partial_k\Gamma^i{}_{jl}-\partial_l\Gamma^i{}_{jk}
  +\Gamma^m{}_{jl}\Gamma^i{}_{mk}-\Gamma^m{}_{jk}\Gamma^i{}_{ml},
\qquad
\operatorname{Ric}_{ij}=R^m{}_{imj}.
$$


In [7]:
def g~i~j : Matrix MathValue := M.inverse g_#_#

def Γ_i_j_k : Tensor MathValue :=
  (1 / 2) * (∂/∂ g_i_k x~j + ∂/∂ g_i_j x~k - ∂/∂ g_j_k x~i)

def Γ~i_j_k : Tensor MathValue := withSymbols [m]
  g~i~m . Γ_m_j_k

def R~i_j_k_l : Tensor MathValue := withSymbols [m]
  ∂/∂ Γ~i_j_l x~k - ∂/∂ Γ~i_j_k x~l
    + Γ~m_j_l . Γ~i_m_k - Γ~m_j_k . Γ~i_m_l

def Ric_i_j : Matrix MathValue := withSymbols [m]
  sum (contract R~m_i_m_j)

def einsteinResidual_i_j : Matrix MathValue := withSymbols [i, j]
  expandAll' (Ric_i_j -' 4 *' g_i_j)


In [8]:
Γ_2_2_4


$\frac{-1}{12}$

In [9]:
Γ_2_4_2


$\frac{-1}{12}$

## 結果の解釈と評価方針

例として取り出した2つの第一種クリストッフェル記号は、単純な成分
$g_{22}=p/6$ から得られ、期待される対称性
$\Gamma_{ijk}=\Gamma_{ikj}$ を示します。完全な検証を表すのが
`einsteinResidual_#_#` です。数学的にはこれは零行列となるため、
スカラー曲率は $5\cdot4=20$ です。

残差の25成分をすべて展開すると、結合ブロックの逆行列と多数の有理式の
簡約が必要になります。そのため、意図的に自動実行される出力セルには
していません。この恒等式を対話的に調べるときは、
`einsteinResidual_2_2` のような個々の成分を評価してください。
